# 02 — Silver Layer
**Person A**

Đọc Bronze → tạo `master_df` → EDA → Feature Engineering → upload Silver

**Input:** Bronze bucket (`bronze-data/instacart/bronze/{BRONZE_RUN_ID}/`)  
**Output:** Silver bucket (`silver-data/instacart/silver/{RUN_ID}/`)
- `master_df` — 32M dòng, dùng xuyên suốt pipeline
- `user_features` — 206k dòng, chưa scale, dùng cho Dashboard
- `user_vector_scaled` — 206k dòng, đã scale, dùng cho Clustering + FAISS

---
⚠️ **Không load `order_products__train`** ở notebook này — giữ lại cho Evaluation.

## Cell 1 — Cài thư viện + clone repo

In [1]:
# Cell 1 — Cài thư viện + clone repo
!pip install supabase -q

import subprocess, sys

subprocess.run([
    "git", "clone",
    "https://github.com/rhyni/retail-recommender-system.git",
    "/kaggle/working/repo"
], check=True)

sys.path.insert(0, "/kaggle/working/repo")
print("✓ Repo đã clone xong")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 7.4 MB/s eta 0:00:00


Cloning into '/kaggle/working/repo'...


✓ Repo đã clone xong


## Cell 2 — Load Secrets + config

In [2]:
from kaggle_secrets import UserSecretsClient
from src import config

secrets = UserSecretsClient()
config.SUPABASE_URL = secrets.get_secret("SUPABASE_URL")
config.SUPABASE_KEY = secrets.get_secret("SUPABASE_KEY")

print("✓ Supabase config loaded")

✓ Supabase config loaded


## Cell 3 — Tạo RUN_ID + khai báo BRONZE_RUN_ID

In [3]:
from datetime import datetime, timezone

RUN_ID        = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
BRONZE_RUN_ID = "20260421T085124Z"  # TODO: copy từ output 01_bronze

print(f"BRONZE_RUN_ID : {BRONZE_RUN_ID}")
print(f"SILVER RUN_ID : {RUN_ID}")

BRONZE_RUN_ID : 20260421T085124Z
SILVER RUN_ID : 20260424T065124Z


## Cell 4 — Load Bronze 

In [4]:
from src.io_utils import load_dataframe

print("Loading Bronze tables...")

prior       = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, BRONZE_RUN_ID, "order_products__prior")
orders      = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, BRONZE_RUN_ID, "orders")
products    = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, BRONZE_RUN_ID, "products")
aisles      = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, BRONZE_RUN_ID, "aisles")
departments = load_dataframe(config.BRONZE_BUCKET, config.BRONZE_PREFIX, BRONZE_RUN_ID, "departments")

PRIOR_ROWS = len(prior)

print(f"\n✓ prior       : {len(prior):>12,} rows")
print(f"✓ orders      : {len(orders):>12,} rows")
print(f"✓ products    : {len(products):>12,} rows")
print(f"✓ aisles      : {len(aisles):>12,} rows")
print(f"✓ departments : {len(departments):>12,} rows")

Loading Bronze tables...
  ✓ loaded [order_products__prior]: 65 chunks | 32,434,489 rows
  ✓ loaded [orders]: 7 chunks | 3,421,083 rows
  ✓ loaded [products]: 1 file | 49,688 rows
  ✓ loaded [aisles]: 1 file | 134 rows
  ✓ loaded [departments]: 1 file | 21 rows

✓ prior       :   32,434,489 rows
✓ orders      :    3,421,083 rows
✓ products    :       49,688 rows
✓ aisles      :          134 rows
✓ departments :           21 rows


## Cell 5 — Validate từng bảng trước khi join

In [5]:
EXPECTED = {
    "prior":       (30_000_000, 34_000_000),
    "orders":      (3_000_000,  3_500_000),
    "products":    (49_000,     50_000),
    "aisles":      (130,        140),
    "departments": (20,         22),
}

for name, df, (lo, hi) in [
    ("prior",       prior,       EXPECTED["prior"]),
    ("orders",      orders,      EXPECTED["orders"]),
    ("products",    products,    EXPECTED["products"]),
    ("aisles",      aisles,      EXPECTED["aisles"]),
    ("departments", departments, EXPECTED["departments"]),
]:
    assert lo <= len(df) <= hi, f"❌ {name}: {len(df):,} ngoài kỳ vọng [{lo:,}, {hi:,}]"
    null_count = df.isnull().sum().sum()
    print(f"  ✓ {name:<25} {len(df):>12,} rows | null={null_count}")

print("\n✅ Tất cả bảng hợp lệ")

  ✓ prior                       32,434,489 rows | null=0
  ✓ orders                       3,421,083 rows | null=206209
  ✓ products                        49,688 rows | null=0
  ✓ aisles                             134 rows | null=0
  ✓ departments                         21 rows | null=0

✅ Tất cả bảng hợp lệ


## Cell 6 — Join → master_df

In [6]:
# Cell 6 — Join → master_df
import pandas as pd

print("Joining tables...")

master_df = (
    prior
    .merge(orders,      on="order_id",      how="left")
    .merge(products,    on="product_id",    how="left")
    .merge(aisles,      on="aisle_id",      how="left")
    .merge(departments, on="department_id", how="left")
)

# Số dòng sau join phải bằng prior
assert len(master_df) == PRIOR_ROWS, (
    f"❌ Join sai: prior={PRIOR_ROWS:,} nhưng master_df={len(master_df):,}"
)

print(f"✓ master_df: {len(master_df):,} rows | {len(master_df.columns)} cols")
print(f"  Columns  : {list(master_df.columns)}")
master_df.head(3)

Joining tables...
✓ master_df: 32,434,489 rows | 15 cols
  Columns  : ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department']


,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry


In [7]:
# # Số order_id unique bị null → phải đúng bằng 206,209
# first_orders = master_df[master_df['is_first_order'] == 1]['order_id'].nunique()
# print(first_orders)  # → phải ra ~206,209

### Cell 7 — Tạo is_first_order + xử lý days_since_prior_order


In [8]:
# Cell 7 — Tạo is_first_order + xử lý days_since_prior_order
null_count = master_df['days_since_prior_order'].isnull().sum()
print(f"Null trong days_since_prior_order: {null_count:,} dòng")

# Đánh dấu đơn đầu tiên
master_df['is_first_order'] = master_df['days_since_prior_order'].isnull().astype('int8')

# fillna(0) để tránh NaN — khi tính feature sẽ filter is_first_order == 0
master_df['days_since_prior_order'] = master_df['days_since_prior_order'].fillna(0).astype('float32')

# Validate: số đơn đầu tiên phải đúng bằng số user
first_order_count = master_df[master_df['is_first_order'] == 1]['order_id'].nunique()
print(f"✓ is_first_order = 1 : {master_df['is_first_order'].sum():,} dòng")
print(f"✓ Đơn đầu tiên unique: {first_order_count:,} (kỳ vọng ~206,209)")

Null trong days_since_prior_order: 2,078,068 dòng
✓ is_first_order = 1 : 2,078,068 dòng
✓ Đơn đầu tiên unique: 206,209 (kỳ vọng ~206,209)


## Cell 8 —  Kiểm tra duplicate

In [9]:
dup_count = master_df.duplicated(subset=['order_id', 'product_id']).sum()
print(f"Duplicate (order_id × product_id): {dup_count:,}")

if dup_count > 0:
    master_df = master_df.drop_duplicates(subset=['order_id', 'product_id'])
    print(f"  → Đã drop, còn lại: {len(master_df):,} dòng")
else:
    print("  → Không có duplicate ✓")

Duplicate (order_id × product_id): 0
  → Không có duplicate ✓


## Cell 9 — Null check toàn bộ cột sau clean


In [10]:
print("Null check toàn bộ cột:")
null_summary = master_df.isnull().sum()
null_cols    = null_summary[null_summary > 0]

if len(null_cols) == 0:
    print("  → Không còn null ✓")
else:
    print(null_cols)

Null check toàn bộ cột:
  → Không còn null ✓


## # Cell 10 — Validate cuối trước khi upload


In [11]:
import pandas as pd

checks = {
    "user_id unique"   : (master_df['user_id'].nunique(),   206_000, 207_000),
    "order_id unique"  : (master_df['order_id'].nunique(),  3_000_000, 3_500_000),
    "product_id unique": (master_df['product_id'].nunique(), 49_000, 50_000),
}

for label, (actual, lo, hi) in checks.items():
    assert lo <= actual <= hi, f"❌ {label}: {actual:,} ngoài kỳ vọng [{lo:,}, {hi:,}]"
    print(f"  ✓ {label:<22}: {actual:,}")

# Memory usage
mem_gb = master_df.memory_usage(deep=True).sum() / 1e9
print(f"\n  ✓ Memory usage: {mem_gb:.2f} GB (kỳ vọng < 8 GB)")
assert mem_gb < 8, f"❌ RAM quá lớn: {mem_gb:.2f} GB"

print("\n✅ master_df sẵn sàng upload")

  ✓ user_id unique        : 206,209
  ✓ order_id unique       : 3,214,874
  ✓ product_id unique     : 49,677

  ✓ Memory usage: 7.22 GB (kỳ vọng < 8 GB)

✅ master_df sẵn sàng upload


In [12]:
# Cell 11 — Upload master_df lên Silver
from src.io_utils import upload_dataframe

upload_dataframe(
    master_df,
    bucket=config.SILVER_BUCKET,
    prefix=config.SILVER_MASTER_PREFIX,
    run_id=RUN_ID,
    table_name="master_df",
)

print(f"✓ master_df uploaded | {len(master_df):,} rows")

  ✓ uploaded [master_df]: 65 chunks | 32,434,489 rows
✓ master_df uploaded | 32,434,489 rows


In [13]:
# Cell 12 — Ghi manifest
from src.supabase_utils import write_manifest

manifest_tables = {
    "master_df": {"rows": len(master_df)}
}

write_manifest(
    bucket=config.SILVER_BUCKET,
    prefix=config.SILVER_MASTER_PREFIX,
    run_id=RUN_ID,
    layer="silver_master",
    created_by="bich",
    tables=manifest_tables,
    source_run_id=BRONZE_RUN_ID,
)

print(f"\n✅ 02a DONE | RUN_ID: {RUN_ID}")
# print("→ Copy RUN_ID này vào SILVER_MASTER_RUN_ID trong 02b")

  ✓ manifest → silver-data/instacart/silver-master/20260424T065124Z/_manifest.json

✅ 02a DONE | RUN_ID: 20260424T065124Z


In [14]:
# Cell 13 — Verify: đọc lại từ Supabase để confirm
from src.io_utils import load_dataframe

verify = load_dataframe(
    config.SILVER_BUCKET,
    config.SILVER_MASTER_PREFIX,
    RUN_ID,
    "master_df",
)

print(f"Verify shape : {verify.shape}")
print(f"Verify cols  : {list(verify.columns)}")
assert len(verify) == len(master_df), "❌ Số dòng sau upload không khớp"
print("✅ Upload verified")

  ✓ loaded [master_df]: 65 chunks | 32,434,489 rows
Verify shape : (32434489, 16)
Verify cols  : ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department', 'is_first_order']
✅ Upload verified
